# পাঠ ১৩ - এজেন্ট মেমরি


## সেটআপ

এই নোটবুকটি দেখায় কিভাবে **Microsoft Agent Framework** (MAF) ব্যবহার করে **স্থায়ী স্মৃতিসম্পন্ন** একটি ট্রাভেল বুকিং এজেন্ট তৈরি করতে হয়।

আপনি শিখবেন কিভাবে বিভিন্ন ধরনের এজেন্ট স্মৃতি — ওয়ার্কিং, শর্ট-টার্ম, এবং লং-টার্ম — কথোপকথনের মাধ্যমে তথ্য ধরে রাখা এবং ব্যবহার করার উপায়কে প্রভাবিত করে।

**প্রয়োজনীয়তা:**
- একটি Azure AI Foundry প্রকল্প যার সাথে একটি চ্যাট মডেল (যেমন `gpt-4o-mini`) ডিপ্লয় আছে।
- Azure CLI তে লগইন করা থাকবেন — আপনার টার্মিনালে `az login` রান করুন।
- `AZURE_AI_PROJECT_ENDPOINT` — আপনার Azure AI Foundry প্রকল্পের এন্ডপয়েন্ট।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## এজেন্ট মেমরির ধরন

এআই এজেন্টরা বিভিন্ন ধরনের মেমরি ব্যবহার করতে পারে, প্রতিটি আলাদা উদ্দেশ্য পরিবেশন করে:

### ওয়ার্কিং মেমরি
আলাপের থ্রেড নিজেই — একটি সেশনে বিনিময় করা মেসেজসমূহ। এজেন্ট একই থ্রেডের পূর্ববর্তী মেসেজগুলোকে রেফার করতে পারে যাতে সঙ্গতি বজায় থাকে। MAF-এ এটি **`agent.create_session()`** এর মাধ্যমে তৈরি হয়, যা একটি `AgentSession` প্রদান করে।

### শর্ট-টার্ম মেমরি
সেই তথ্য যা একটি কাজ বা সেশনের সময়কাল ধরে থাকে কিন্তু স্থায়ীভাবে সংরক্ষণ করা হয় না। উদাহরণস্বরূপ, এজেন্ট একটি বহুখণ্ড পরিকল্পনা আলাপের সময় তথ্য সংগ্রহ করতে পারে এবং সেগুলো ব্যবহার করে চূড়ান্ত সূচিপত্র তৈরি করতে পারে।

### লং-টার্ম মেমরি
রুচি ও তথ্য যা **সেশনের বাইরেও** বজায় থাকে। একজন ফিরে আসা ব্যবহারকারীকে তার খাদ্য সংক্রান্ত বিধিনিষেধ বা ভ্রমণ শৈলী পুনরাবৃত্তি করতে হয় না। লং-টার্ম মেমরি সাধারণত একটি বাহ্যিক সংরক্ষণাগার দ্বারা সমর্থিত থাকে — একটি ডাটাবেস, ফাইল, বা ভেক্টর ইনডেক্স — এবং টুলগুলোর মাধ্যমে এজেন্টের সামনে উপস্থাপন করা হয়।


## সেশনস সহ ওয়ার্কিং মেমোরি

স্মৃতির সবচেয়ে সহজরূপ হল কথোপকথন সেশন। যখন আপনি একই সেশন (`agent.create_session()` এর মাধ্যমে তৈরি) ধারাবাহিক `agent.run()` কলগুলিতে ব্যবহার করেন, তখন এজেন্ট ঐ কথোপকথনের পূর্ণ ইতিহাস দেখতে পায় এবং পূর্বের বিবরণগুলি স্মরণ করতে পারে।

চলুন একটি ট্রাভেল এজেন্ট তৈরি করি এবং ওয়ার্কিং মেমোরি প্রদর্শন করি।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

এজেন্ট সঠিকভাবে বাজেট স্মরণ করেছে কারণ উভয় মেসেজ একই সেশন শেয়ার করে। এটিই **ওয়ার্কিং মেমোরি** — এটি শুধুমাত্র সেশনের জীবদ্দশার জন্য বিদ্যমান। 

### নতুন থ্রেড হলে কী হয়?

যদি আমরা একটি **নতুন** সেশন তৈরি করি, তাহলে এজেন্ট পূর্ববর্তী কথোপকথনের কোনো স্মৃতি রাখে না:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## দীর্ঘমেয়াদী স্মৃতি প্যাটার্ন

**সেশনের মধ্যে** ব্যবহারকারীর পছন্দগুলি মনে রাখতে, আমাদের একটি স্থায়ী স্টোর দরকার যা কথোপকথনের থ্রেডের বাইরে থাকে। এজেন্ট এই স্টোরে অ্যাক্সেস করে **টুলস** এর মাধ্যমে — এমন ফাংশন যা এটি তথ্য সংরক্ষণ এবং পুনরুদ্ধার করার জন্য কল করতে পারে।

নিচে আমরা একটি সহজ ইন-মেমরি পছন্দ স্টোর বাস্তবায়ন করেছি (প্রোডাকশনে আপনি এটি একটি ডাটাবেস বা ভেক্টর ইনডেক্সের সাথে ব্যাক করবেন) এবং এটি টুলস হিসাবে প্রকাশ করেছি যা এজেন্ট ব্যবহার করতে পারে।

### স্থাপত্য
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — প্রথমবারের মতো ব্যবহারকারী একটি বার্ষিকী ট্রিপ বুক করে

সারা প্রথমবার ভ্রমণ করছেন। এজেন্টকে তার পছন্দসমূহ টুলসের মাধ্যমে সংরক্ষণ করতে হবে এবং সেগুলি ব্যবহার করে হোটেলগুলোর সুপারিশ করতে হবে।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### দৃশ্যপট ২ — সারা কয়েক সপ্তাহ পরে ফিরে আসে

সারা একটি **নতুন থ্রেড** শুরু করেন (নতুন সেশন অনুকরণ)। কর্মরত মেমরি খালি, কিন্তু দীর্ঘমেয়াদি পছন্দ সংরক্ষণে তার তথ্য এখনও রয়েছে। এজেন্টকে তা উদ্ধার করে ব্যক্তিগতকৃত সুপারিশের জন্য ব্যবহার করা উচিত।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারসংক্ষেপ

এই পাঠে আপনি তিন ধরনের এজেন্ট মেমোরি এবং Microsoft Agent Framework দিয়ে সেগুলো কীভাবে বাস্তবায়ন করা যায় তা শিখেছেন:

| মেমোরির ধরন | MAF পদ্ধতি | আজীবনকাল |
|---|---|---|
| **কর্মক্ষম** | `agent.create_session()` | একক আলাপচারিতা |
| **স্বল্পকালীন** | একটি থ্রেডের মধ্যে সঞ্চিত প্রসঙ্গ | একক কাজ / সেশন |
| **দীর্ঘকালীন** | `@tool` ফাংশনের মাধ্যমে অ্যাক্সেস করা বাহ্যিক দোকান | সেশন জুড়ে |

### মূল বিষয়গুলি
1. **`agent.create_session()`** কর্মক্ষম মেমোরি প্রদান করে — এজেন্ট একটি সেশনের মধ্যে সম্পূর্ণ আলাপচারিতা ইতিহাস দেখতে পারে।
2. **নতুন সেশনগুলো প্রসঙ্গ হারায়** — দীর্ঘকালীন মেমোরি ছাড়া এজেন্ট অতীত আলাপচারিতা মনে রাখতে পারে না।
3. **`@tool` ফাংশনগুলো** ফাঁক পূরণ করে — এগুলো এজেন্টকে তথ্য সংরক্ষণ এবং পুনরুদ্ধার করতে দেয় একটি স্থায়ী দোকান থেকে।
4. **ব্যক্তিগতকরণ সময়ের সাথে উন্নত হয়** — যত বেশি পছন্দসই তথ্য সংরক্ষিত হয়, এজেন্টের সুপারিশ তত ভালো হয়।

### বাস্তব জীবনের প্রয়োগ
- **কাস্টমার সার্ভিস**: গ্রাহকের ইতিহাস এবং পছন্দ মনে রাখা
- **ব্যক্তিগত সহকারী**: দিন বা সপ্তাহ জুড়ে প্রসঙ্গ বজায় রাখা
- **স্বাস্থ্যসেবা**: রোগীর তথ্য এবং পছন্দ অনুসরণ করা
- **ই-কমার্স**: ইতিহাসের ভিত্তিতে ব্যক্তিগতকৃত কেনাকাটা

### পরবর্তী পদক্ষেপসমূহ
- ইন-মেমোরি ডিক্টের পরিবর্তে ডাটাবেস বা ভেক্টর স্টোর ব্যবহার করুন (যেমন Azure AI Search)
- সময় সংবেদনশীল তথ্যের জন্য মেমোরি মেয়াদ শেষ করার ব্যবস্থা যোগ করুন
- ভাগ করা মেমোরি সহ মাল্টি-এজেন্ট সিস্টেম তৈরি করুন
- নলেজ-গ্রাফ সমর্থিত মেমোরির জন্য Cognee নোটবুক অন্বেষণ করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**দায়িত্ব অস্বীকার**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাসাধ্য সঠিকতা নিশ্চিত করার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসম্পূর্ণতা থাকতে পারে। মূল নথির দেশীয় ভাষার সংস্করণই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশয়। এই অনুবাদের ব্যবহারে কোনো ভুল বোঝাবুঝি বা ভ্রান্ত ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
